# Part 4 · Agentics Vision Demo

**The story:** the first three parts secured *service* traffic with ambient. This part is about *agents*. An engineering org wants developers to ship agents fast, but only from **approved** building blocks, deployed to whichever runtime the platform team allows, with the tools each agent can call governed centrally. AgentRegistry is that control plane: a catalog of approved MCP tool servers, skills and runtimes; a CLI (`arctl`) to scaffold, build and publish an agent; and a one-line deploy to a connected runtime. Governance rides the same agentgateway data plane you already run.

This part runs on **`mesh1` only** and stands alone. Everything is behind one Keycloak IdP, and the same agentgateway that fronts the platform enforces tool-level access at an ambient waypoint.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 280" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="740" height="280" rx="10" fill="#f8fafc"/>
  <text x="370" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">AgentRegistry: the governed catalog and deploy hub</text>
  <defs><marker id="ra" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>

  <!-- developer + arctl -->
  <rect x="16" y="108" width="104" height="60" rx="9" fill="#e2e8f0" stroke="#64748b"/>
  <text x="68" y="132" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">developer</text>
  <text x="68" y="150" text-anchor="middle" font-size="10.5" fill="#475569">arctl CLI</text>
  <text x="192" y="122" text-anchor="middle" font-size="9" fill="#475569">init · build · publish</text>
  <line x1="120" y1="138" x2="266" y2="138" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>

  <!-- registry -->
  <rect x="266" y="60" width="210" height="158" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/>
  <text x="371" y="84" text-anchor="middle" font-size="13" font-weight="700" fill="#312e81">AgentRegistry</text>
  <text x="371" y="99" text-anchor="middle" font-size="9.5" fill="#4338ca">in-cluster control plane</text>
  <rect x="284" y="112" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="129" text-anchor="middle" font-size="10.5" fill="#312e81">approved MCP tool servers</text>
  <rect x="284" y="144" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="161" text-anchor="middle" font-size="10.5" fill="#312e81">skills</text>
  <rect x="284" y="176" width="174" height="26" rx="6" fill="#eef2ff" stroke="#a5b4fc"/>
  <text x="371" y="193" text-anchor="middle" font-size="10.5" fill="#312e81">runtimes</text>
  <text x="540" y="96" text-anchor="middle" font-size="9" fill="#475569">deploy</text>
  <line x1="476" y1="110" x2="576" y2="110" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>
  <line x1="476" y1="168" x2="576" y2="168" stroke="#334155" stroke-width="1.8" marker-end="url(#ra)"/>

  <!-- runtimes -->
  <rect x="576" y="92" width="150" height="36" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="651" y="110" text-anchor="middle" font-size="11.5" font-weight="600" fill="#14532d">kagent (mesh1)</text>
  <text x="651" y="123" text-anchor="middle" font-size="9" fill="#166534">local runtime</text>
  <rect x="576" y="150" width="150" height="36" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
  <text x="651" y="168" text-anchor="middle" font-size="11.5" font-weight="600" fill="#7c2d12">AWS AgentCore</text>
  <text x="651" y="181" text-anchor="middle" font-size="9" fill="#92400e">Bedrock runtime</text>

  <text x="370" y="252" text-anchor="middle" font-size="11" fill="#64748b">A developer scaffolds an agent from approved catalog tools; the registry publishes it and deploys the same agent to any runtime.</text>
</svg></div>

**What you'll run:** browse the approved catalog, scaffold an agent wired to one approved tool, build and publish it, deploy it to kagent and ask it a question, add a second approved tool and watch it appear, then lock the agent down with an AccessPolicy, turn a REST API into MCP tools with no code, and (optionally, with your own AWS session) deploy the *same* agent to AWS Bedrock AgentCore.

## Open the consoles

The platform is reachable from your laptop over the mesh1 LoadBalancer IP via `sslip.io` hostnames (no `/etc/hosts` edits). `Connect` below prints the exact URLs for this cluster.

- **AgentRegistry UI** — the catalog, runtimes, deployments and Access Policies.
- **kagent UI** — the Solo Enterprise UI for the runtime: chat with deployed agents, watch **Tracing** span trees, manage Access Policies. Same login.
- **Keycloak** — the IdP (realm `agentregistry`); log in as `admin-user` / `password`.
- **Swagger Petstore** — https://petstore.swagger.io, the public REST API we turn into MCP tools.

In [ ]:
# Connect: load the mesh1 platform facts, put arctl on PATH, and log the CLI in to
# the in-cluster AgentRegistry as admin-user (Keycloak group admins -> superuser).
cd "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/agentregistry"
source scripts/connect.sh
echo
echo "  == consoles for this cluster =="
printf "  %-20s %s\n" "AgentRegistry UI:" "http://${AR_HOST}"
printf "  %-20s %s\n" "kagent UI:"        "http://${KAGENT_UI_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "Keycloak:"         "http://${KEYCLOAK_HOST}  (admin-user / password)"
printf "  %-20s %s\n" "Swagger Petstore:" "https://petstore.swagger.io"

## Reset the cluster to a clean demo state

Only Part 4 needs this platform, so between demos it sits **parked at 0 replicas** to free about 1.9 GiB on `mesh1`. This cell brings it back up (waiting for the registry and IdP), then removes what *this* demo creates: the `agentdemo` agent and its kagent deployment, the deployed MCP tool servers, any AccessPolicy + waypoint label, and the Petstore OpenAPI backend. The platform (kagent, AgentRegistry, Keycloak) and the approved catalog stay up, so the demo is a clean read and Parts 1-3 (which use their own namespaces) are untouched.

Run it **before** a fresh run. When you're done with Part 4 and want the memory back, park the platform again with `bash scripts/reset.sh down` from `istio-ambient-demo-kind/demo-scripts/agentregistry`.

In [ ]:
cd "$(git rev-parse --show-toplevel)/istio-ambient-demo-kind/demo-scripts/agentregistry"
source scripts/connect.sh 2>/dev/null

# Demos 1-3 don't need Part 4's platform, so it's parked at 0 replicas between demos
# to free ~1.9 GiB (AgentRegistry + ClickHouse/Postgres, ar-keycloak, kagent, kyverno).
# The kagent Enterprise UI (solo-cost) stays up — demo-7 shares it.
# Bring it back up and wait for the registry + IdP BEFORE reset.sh runs: reset purges the
# catalog via arctl, so the server must be live first, or a re-run hits "already exists".
for ns in agentregistry-system ar-keycloak kagent kyverno; do
  kubectl --context "$CTX" -n "$ns" scale deploy,statefulset --all --replicas=1 >/dev/null 2>&1 || true
done
kubectl --context "$CTX" -n ar-keycloak          rollout status statefulset/keycloak                   --timeout=300s
kubectl --context "$CTX" -n agentregistry-system rollout status deploy/agentregistry-enterprise-server --timeout=300s
kubectl --context "$CTX" -n kagent               rollout status deploy/kagent-controller               --timeout=300s

rm -f "${TMPDIR:-/tmp}"/agentcore-deploy.log*   # stale logs from an earlier AWS push
bash scripts/reset.sh


## 1. Browse the registry: approved tools, skills and runtimes

A developer starts here. The catalog holds only what the platform team has approved: two MCP tool servers, one skill, and the runtimes an agent may deploy to.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 244" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="244" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">1 · The approved catalog: tools, skills and runtimes</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="98" width="112" height="54" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="72" y="118" text-anchor="middle" font-size="10.5" font-weight="700" fill="#334155">developer</text><text x="72" y="134" text-anchor="middle" font-size="8" fill="#475569">browses the catalog</text><line x1="128" y1="125" x2="186" y2="125" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><rect x="190" y="50" width="514" height="152" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.6"/><text x="447" y="70" text-anchor="middle" font-size="10" font-weight="700" fill="#312e81">AgentRegistry catalog · only what the platform approved</text><line x1="360" y1="82" x2="360" y2="196" stroke="#c7d2fe"/><line x1="500" y1="82" x2="500" y2="196" stroke="#c7d2fe"/><text x="277" y="98" text-anchor="middle" font-size="9" font-weight="700" fill="#4338ca">MCP tool servers</text><rect x="202" y="111" width="150" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="277" y="128" text-anchor="middle" font-size="8.5" fill="#1e293b">my-mcp</text><rect x="198" y="141" width="158" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="277" y="158" text-anchor="middle" font-size="8.5" fill="#1e293b">everything-server</text><text x="432" y="98" text-anchor="middle" font-size="9" font-weight="700" fill="#4338ca">Skills</text><rect x="377" y="111" width="110" height="24" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1"/><text x="432" y="128" text-anchor="middle" font-size="8.5" fill="#1e293b">dice-game</text><text x="602" y="98" text-anchor="middle" font-size="9" font-weight="700" fill="#4338ca">Runtimes</text><rect x="527" y="111" width="150" height="24" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="602" y="128" text-anchor="middle" font-size="8.5" fill="#1e293b">kind-kagent</text><rect x="527" y="141" width="150" height="24" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="602" y="158" text-anchor="middle" font-size="8.5" fill="#1e293b">aws-agentcore</text><text x="360" y="224" text-anchor="middle" font-size="10.5" fill="#64748b">The developer only ever sees approved building blocks: MCP tool servers, skills, and the runtimes the platform team allows.</text></svg></div>

In [ ]:
echo "  == approved MCP tool servers =="; arctl get mcpservers
echo; echo "  == approved skills ==";          arctl get skills
echo; echo "  == connected runtimes ==";        arctl get runtimes

## 2. Scaffold the agent, wired to ONE approved tool

`arctl init` scaffolds a complete, working agent project (ADK + Python): the agent code, its `agent.yaml` record, the Dockerfile, everything. The flags are the governance story — `--mcp my-mcp@latest` wires it to an **approved** MCP tool server from the catalog, and `--model-provider`/`--model-name` pin the model. `--output-dir` drops the project at the lab root, so watch **`agentdemo/`** appear in your editor next to these notebooks.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 228" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="228" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">2 · arctl init: scaffold an agent from approved building blocks</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="74" width="180" height="116" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.6"/><text x="106" y="94" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">catalog (approved)</text><rect x="31" y="109" width="150" height="24" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1"/><text x="106" y="126" text-anchor="middle" font-size="8.5" fill="#1e293b">my-mcp  ✓</text><rect x="31" y="145" width="150" height="24" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1"/><text x="106" y="162" text-anchor="middle" font-size="8.5" fill="#1e293b">dice-game  ✓</text><line x1="196" y1="132" x2="300" y2="132" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><text x="248" y="124" text-anchor="middle" font-size="8.5" font-weight="700" fill="#475569">arctl init</text><rect x="302" y="80" width="258" height="104" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="431" y="102" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">agentdemo</text><text x="431" y="120" text-anchor="middle" font-size="9" fill="#166534">ADK + Python project</text><text x="431" y="138" text-anchor="middle" font-size="9" fill="#166534">tool: my-mcp</text><text x="431" y="154" text-anchor="middle" font-size="9" fill="#166534">skill: dice-game</text><text x="431" y="172" text-anchor="middle" font-size="8" fill="#166534">(scaffolded, runnable)</text><text x="360" y="210" text-anchor="middle" font-size="10.5" fill="#64748b">arctl init scaffolds a working ADK + Python agent, wired to the approved my-mcp tool server and the approved dice-game skill.</text></svg></div>

In [ ]:
arctl init agent agentdemo --framework adk --language python \
  --model-provider anthropic --model-name claude-haiku-4-5 \
  --mcp my-mcp@latest --output-dir "$PROJECT_ROOT"

### Add the approved dice-game skill from the registry

A **skill** in the catalog is approved, versioned *guidance* — a `SKILL.md` with recorded source provenance. This one tells any agent how to run a fair dice game: always roll with `roll_die`, use `sum` for arithmetic instead of doing it in-head, `check_prime` for primality, and report in a fixed house format. Look at the approved record, then pull its source straight from the registry — watch **`dice-game/`** appear next to `agentdemo/`, and open `dice-game/SKILL.md` to read the guidance itself:

In [ ]:
arctl get skill dice-game -o yaml
echo
arctl pull skill dice-game "$PROJECT_ROOT/dice-game"

Every scaffolded agent ships a **prompt loader**: `build_instruction()` in `agent.py` assembles the system instruction from `prompts.json` next to the module (the same file the platform mounts at `/config` on managed runtimes). Hand it the skill body — front matter stripped, JSON-wrapped — and the approved guidance becomes the agent's system instruction at the next build:

In [ ]:
awk '/^---$/{c++;next} c>=2' "$PROJECT_ROOT/dice-game/SKILL.md" \
  | jq -Rs '[{name:"dice-game", content:.}]' > "$PROJECT_ROOT/agentdemo/agentdemo/prompts.json"

echo "  == the agent's prompts.json now carries the skill =="
jq -r '.[0].content' "$PROJECT_ROOT/agentdemo/agentdemo/prompts.json" | sed -n '2,8p'

## 3. Build it and chat with it on your laptop

`arctl build` builds the agent container from the scaffold. Before it goes anywhere near a cluster, run it **locally** — the same image, on your laptop. The cell also drops the model key into the agent's `.env` (borrowed from the cluster's `kagent-anthropic` Secret, so no key is typed or pasted).

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">3 · Build and push the image, publish the record</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="92" width="150" height="66" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="91" y="116" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">agentdemo</text><text x="91" y="132" text-anchor="middle" font-size="8.5" fill="#166534">project (ADK)</text><text x="91" y="147" text-anchor="middle" font-size="8.5" fill="#166534">source on disk</text><line x1="166" y1="110" x2="336" y2="86" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><text x="250" y="92" text-anchor="middle" font-size="8.5" font-weight="700" fill="#475569">arctl build --push</text><rect x="338" y="64" width="366" height="46" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="521" y="86" text-anchor="middle" font-size="10" font-weight="700" fill="#334155">localhost:5001/agentdemo:latest</text><text x="521" y="101" text-anchor="middle" font-size="8" fill="#475569">image in the cluster registry</text><line x1="166" y1="140" x2="336" y2="166" stroke="#2563eb" stroke-width="1.8" marker-end="url(#b)"/><text x="250" y="166" text-anchor="middle" font-size="8.5" font-weight="700" fill="#1d4ed8">arctl apply</text><rect x="338" y="144" width="366" height="46" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.4"/><text x="521" y="166" text-anchor="middle" font-size="10" font-weight="700" fill="#312e81">Agent record</text><text x="521" y="181" text-anchor="middle" font-size="8" fill="#4338ca">published to the AgentRegistry catalog</text><text x="360" y="214" text-anchor="middle" font-size="10.5" fill="#64748b">arctl build --push containerizes the agent to the cluster registry; arctl apply publishes the Agent record to the catalog.</text></svg></div>

In [ ]:
arctl build "$PROJECT_ROOT/agentdemo"

# give the local run the model key: borrow it from the cluster's Secret
KEY="$(kc -n kagent get secret kagent-anthropic -o jsonpath='{.data.ANTHROPIC_API_KEY}' | base64 -d)"
sed -i '' "s|^ANTHROPIC_API_KEY=.*|ANTHROPIC_API_KEY=${KEY}|" "$PROJECT_ROOT/agentdemo/.env"
echo "✓ built; .env carries the model key — run it in a terminal (next cell)"

`arctl run` is interactive, so copy this into a **terminal** — it serves the agent on `localhost:8080` and opens a chat (from the `istio-ambient-demo-kind/` folder):

```bash
arctl run ./agentdemo
```

Ask it (paste into the chat):

```text
Roll a 20-sided die and tell me whether the result is prime.
```

It answers in the dice-game house format, using the local `roll_die` + `check_prime` tools — the skill is in the image you just built. The approved catalog tool (`word_count` from `my-mcp`) comes online when the agent is **deployed**: the registry stands the MCP server up next to it, in §4.

> If the chat fails with `404 ... route not found`, another lab's port-forward is holding `:8080` — find it with `lsof -iTCP:8080 -sTCP:LISTEN` and kill the stray `kubectl port-forward`. `Ctrl-C` the chat when you're done.


### From your laptop to the catalog

Everything so far is local: a folder on your machine, an image in your local Docker, a chat on `localhost`. Nothing is governed yet — no other developer can find this agent, no runtime can pull it, no policy applies to it. Two commands change that:

- `arctl build --push` pushes the image to the **cluster's registry**, where the connected runtimes can pull it.
- `arctl apply -f agent.yaml` publishes the **Agent record** to the catalog: the agent becomes a versioned, discoverable entry — its model, its image, its approved tool list — deployable to any runtime the platform team has connected, and governable like everything else in the registry.

From here on, the registry owns the agent; the folder on your laptop is just source.

In [ ]:
arctl build "$PROJECT_ROOT/agentdemo" --push
echo; echo "  == publish the Agent to the catalog =="
arctl apply -f "$PROJECT_ROOT/agentdemo/agent.yaml"

The catalog now lists it:

In [ ]:
arctl get agents

## 3b. Push the same agent to AWS Bedrock AgentCore, in the background

The point of the registry is that one agent record deploys to whatever runtime the platform allows. The `agentdemo` you just published deploys to **AWS Bedrock AgentCore** as a second runtime, no rewrite. And it starts now, in the background, so it is provisioned by the time you reach §9.

The cell signs in to AWS and hands the in-cluster registry your credentials in the **foreground** (~30-60s; this must finish before the next step talks to the registry), then runs the slow part in the **background**: on a first run it connects the `aws-agentcore` runtime (cross-account role + ECR), then builds the image for AgentCore, pushes it to ECR, pushes the agent source to git (AgentCore clones it at deploy time), and deploys. AgentCore publishes its **own** agent record (`agentdemo-agentcore`, ECR image), so it never touches the kagent `agentdemo`.

Needs `AWS_PROFILE` and `AGENT_GIT_URL` (from `.env.aws` next to the scripts, or your `SECRETS_FILE`). Watch it any time: `tail -f /tmp/agentcore-deploy.log`.

In [ ]:
source scripts/agentcore.sh

## 4. Deploy onto kagent and ask it a question

One `arctl apply` deploys the MCP tool server, one deploys the agent. The registry binds them to the `kind-kagent` runtime and derives the agent's tool list from the MCP servers on that runtime. The agent deploys **keyless**: a Kyverno policy injects the Anthropic key from a Secret when the Agent is created, so the key is never in a manifest or the pod spec.

> `ask.sh` mints an OIDC token *inside* the cluster (as `admin-user`) and calls the agent's A2A endpoint through kagent, so the ask behaves the same in a terminal and in a cell.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 264" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="264" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">4 · Deploy to the kind-kagent runtime, then ask it</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="70" width="132" height="60" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.4"/><text x="82" y="92" text-anchor="middle" font-size="9.5" font-weight="700" fill="#312e81">AgentRegistry</text><text x="82" y="108" text-anchor="middle" font-size="8.5" fill="#4338ca">Agent record</text><text x="82" y="122" text-anchor="middle" font-size="8.5" fill="#4338ca">+ my-mcp</text><line x1="148" y1="100" x2="236" y2="100" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><text x="192" y="92" text-anchor="middle" font-size="8" font-weight="700" fill="#475569">arctl apply</text><rect x="238" y="52" width="300" height="110" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="388" y="72" text-anchor="middle" font-size="9.5" font-weight="700" fill="#14532d">kagent · runtime kind-kagent</text><rect x="278" y="85" width="220" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="388" y="102" text-anchor="middle" font-size="8.5" fill="#1e293b">MCPServer  my-mcp</text><rect x="278" y="117" width="220" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="388" y="134" text-anchor="middle" font-size="8.5" fill="#1e293b">Agent  agentdemo (keyless)</text><rect x="586" y="74" width="118" height="66" rx="8" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.4"/><text x="645" y="96" text-anchor="middle" font-size="9.5" font-weight="700" fill="#5b21b6">Kyverno</text><text x="645" y="112" text-anchor="middle" font-size="8" fill="#6d28d9">injects ANTHROPIC</text><text x="645" y="125" text-anchor="middle" font-size="8" fill="#6d28d9">_API_KEY from Secret</text><line x1="586" y1="120" x2="510" y2="124" stroke="#2563eb" stroke-width="1.8" stroke-dasharray="5 3" marker-end="url(#b)"/><text x="548" y="113" text-anchor="middle" font-size="7.5" fill="#6d28d9">key</text><rect x="16" y="196" width="132" height="44" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.4"/><text x="82" y="214" text-anchor="middle" font-size="10" font-weight="700" fill="#334155">ask.sh</text><text x="82" y="229" text-anchor="middle" font-size="7.5" fill="#475569">OIDC token (admin-user)</text><line x1="148" y1="218" x2="320" y2="218" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><text x="234" y="210" text-anchor="middle" font-size="8" font-weight="700" fill="#166534">A2A request</text><rect x="322" y="196" width="240" height="44" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="442" y="214" text-anchor="middle" font-size="10" font-weight="700" fill="#14532d">agentdemo answers</text><text x="442" y="229" text-anchor="middle" font-size="8" fill="#166534">calls my-mcp, returns the result</text><text x="360" y="256" text-anchor="middle" font-size="10.5" fill="#64748b">One apply deploys the MCP server and the agent to kagent, keyless (Kyverno injects the key). ask.sh calls the agent's A2A endpoint.</text></svg></div>

In [ ]:
echo "  == deploy the my-mcp tool server =="
arctl apply -f yaml/deploy-mcp-my-mcp.yaml
until kc -n kagent get mcpserver/my-mcp >/dev/null 2>&1; do sleep 2; done
kc -n kagent wait --for=condition=Ready mcpserver/my-mcp --timeout=180s

echo; echo "  == deploy the agent =="
arctl apply -f yaml/deploy-kagent.yaml
# the registry creates the kagent Deployment asynchronously, a moment after apply
# returns, so wait for it to EXIST before waiting on its rollout.
until kc -n kagent get deploy/agentdemo >/dev/null 2>&1; do sleep 2; done
kc -n kagent rollout status deploy/agentdemo --timeout=180s
echo; kc -n kagent get pods | grep -E 'NAME|agentdemo|my-mcp'

<details>
<summary><strong>What got created (click to expand): the kagent objects arctl deployed, keyless</strong></summary>

You ran `arctl apply`, not `kubectl`. The registry turned the two deployment manifests into Kubernetes objects in the `kagent` namespace.

`MCPServer/my-mcp` (from `deploy-mcp-my-mcp.yaml`):
```yaml
apiVersion: kagent.dev/v1alpha1
kind: MCPServer
metadata:
  name: my-mcp                       # namespace: kagent
  labels: { kagent.solo.io/waypoint: "true" }
spec:
  transportType: http
  httpTransport: { path: /mcp, targetPort: 3000 }
  deployment: { image: localhost:5001/my-mcp:latest, port: 3000, replicas: 1 }
```

`Agent/agentdemo` (from `deploy-kagent.yaml`), published **keyless**:
```yaml
apiVersion: kagent.dev/v1alpha2
kind: Agent
metadata:
  name: agentdemo                    # namespace: kagent
  labels: { kagent.solo.io/waypoint: "true" }
spec:
  type: BYO
  byo:
    deployment:
      image: localhost:5001/agentdemo:latest
      env:
      - { name: MCP_SERVERS_CONFIG, value: '[{"name":"my-mcp","type":"remote","url":"http://my-mcp.kagent.svc.cluster.local:3000/mcp"}]' }
      - name: ANTHROPIC_API_KEY          # NOT in the manifest; added by Kyverno
        valueFrom:
          secretKeyRef: { name: kagent-anthropic, key: ANTHROPIC_API_KEY }
```

The `ANTHROPIC_API_KEY` line is not in `deploy-kagent.yaml`. A Kyverno `ClusterPolicy/inject-agent-model-key` mutates every BYO Agent in `kagent` at admission and adds the secret reference, so the key is never written into a manifest or the pod spec:

```yaml
# ClusterPolicy/inject-agent-model-key  (category: AgentRegistry)
match:         kinds [kagent.dev/v1alpha2/Agent], namespaces [kagent]
preconditions: spec.type == "BYO"  AND  no ANTHROPIC_API_KEY env present
mutate:        add /spec/byo/deployment/env/-  ->  ANTHROPIC_API_KEY from Secret/kagent-anthropic
```

Because the MCPServer and the Agent both carry `kagent.solo.io/waypoint: "true"`, the controller also auto-provisions an agentgateway **waypoint** (Gateway + HTTPRoute + Deployment) in front of each, ready for the AccessPolicy in §6:

```
$ kubectl -n kagent get gateway
mcpserver-my-mcp-waypoint    enterprise-agentgateway-waypoint  Programmed
agent-agentdemo-waypoint     enterprise-agentgateway-waypoint  Programmed
```

(§5's `everything-server` is the same `MCPServer` shape as `my-mcp`.)

See it live:
```
kubectl --context kind-mesh1 -n kagent get mcpserver,agent,deploy,gateway,httproute
kubectl --context kind-mesh1 -n kagent get agent agentdemo -o yaml
kubectl --context kind-mesh1 get clusterpolicy inject-agent-model-key -o yaml
```

</details>

In [ ]:
echo "  == ask the agent which tools it has =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

### Now run a real task and watch the trace

Listing tools is one thing, now watch the agent actually **call** them. Give `agentdemo` a multi-step task on the CLI (or chat in the kagent UI). `ask.sh` prints the tool-call trace pulled from the A2A response, so you see each hop and its result inline (`roll_die` -> `check_prime`). Notice the middle step: the agent has no calculator tool yet, so it does the addition itself — §5 approves one and the same task changes shape.

In [ ]:
./scripts/ask.sh "Roll a 13-sided die, add 48291 to the result, then tell me if it is prime."

Now open the **kagent UI** (the `Connect` cell printed the URL; log in as `admin-user` / `password`) → **Tracing**. The run shows up as a span tree — `invocation → call_llm → generate_content (claude-haiku) → execute_tool roll_die → check_prime` — with the model and token usage on each LLM span. **Agents → agentdemo** chats with the same agent in the browser; every chat lands in Tracing too.

## 5. Add a second approved tool, watch it appear

`agentdemo` was scaffolded with one tool (`my-mcp`). The agent's tools are **declared** in `agent.yaml`, so you add another approved server by declaring it and re-applying — in the **AgentRegistry UI** (Catalog → `agentdemo` → **Edit** → add `everything-server`), or by pasting the snippet below into `agentdemo/agent.yaml` under `spec.mcpServers:`.

Then run the cell after it to apply, deploy the MCP server, and re-deploy the agent — all `arctl`, no rebuild.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 232" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="232" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">5 · Approve a second tool, re-derive the agent's tool list</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="16" y="86" width="170" height="80" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="101" y="108" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">agentdemo</text><text x="101" y="126" text-anchor="middle" font-size="8.5" fill="#166534">tools so far:</text><rect x="26" y="135" width="150" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="101" y="152" text-anchor="middle" font-size="8.5" fill="#1e293b">my-mcp</text><rect x="214" y="64" width="150" height="60" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="289" y="86" text-anchor="middle" font-size="9.5" font-weight="700" fill="#7c2d12">everything-server</text><text x="289" y="102" text-anchor="middle" font-size="7.5" fill="#92400e">sum · echo · printenv</text><text x="289" y="114" text-anchor="middle" font-size="7.5" fill="#92400e">reverse_text · to_uppercase</text><line x1="289" y1="124" x2="360" y2="150" stroke="#d97706" stroke-width="1.8" marker-end="url(#d)"/><text x="300" y="140" text-anchor="middle" font-size="7.5" font-weight="700" fill="#92400e">add to mcpServers</text><line x1="186" y1="150" x2="360" y2="150" stroke="#334155" stroke-width="1.8" marker-end="url(#n)"/><rect x="362" y="86" width="342" height="80" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/><text x="533" y="108" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">agentdemo (redeployed)</text><text x="533" y="126" text-anchor="middle" font-size="8.5" fill="#166534">tool list re-derived:</text><rect x="387" y="135" width="150" height="24" rx="6" fill="#dbeafe" stroke="#60a5fa" stroke-width="1"/><text x="462" y="152" text-anchor="middle" font-size="8.5" fill="#1e293b">my-mcp</text><rect x="543" y="135" width="150" height="24" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1"/><text x="618" y="152" text-anchor="middle" font-size="8.5" fill="#1e293b">everything-server</text><text x="360" y="212" text-anchor="middle" font-size="10.5" fill="#64748b">Add the approved everything-server to the agent's mcpServers and redeploy; the registry re-derives the tool list, so the new tools appear.</text></svg></div>

In [ ]:
arctl apply -f "$PROJECT_ROOT/agentdemo/agent.yaml"
arctl apply -f yaml/deploy-mcp-everything-server.yaml


until kc -n kagent get mcpserver/everything-server >/dev/null 2>&1; do sleep 2; done
kc -n kagent wait --for=condition=Ready mcpserver/everything-server --timeout=180s

arctl delete deployment agentdemo >/dev/null 2>&1 || true
arctl apply -f yaml/deploy-kagent.yaml || { sleep 5; arctl apply -f yaml/deploy-kagent.yaml; }
until kc -n kagent get deploy/agentdemo >/dev/null 2>&1; do sleep 2; done
kc -n kagent rollout status deploy/agentdemo --timeout=180s

In [ ]:
echo "  == tools now (the everything-server tools appear) =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

### The same task, through the new tool

Run the identical prompt again. The trace changes shape: the addition now goes through the approved MCP tool (`roll_die` -> `everything_server_sum` -> `check_prime`). Nothing about the agent was rewritten; a tool was approved and the agent started using it.

In [ ]:
./scripts/ask.sh "Roll a 13-sided die, add 48291 to the result, then tell me if it is prime."

## 6. Govern the tools with an AccessPolicy

The agent can call everything on `everything-server`, including `printenv`. A platform owner cuts that down to least privilege: an **allowlist** naming only `sum`. It is enforced at an **agentgateway waypoint** in front of the MCP server, on the agent's own certificate identity, and it filters the tool from the list, so a denied tool cannot even be seen, let alone called.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/>
  <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Least privilege: an AccessPolicy governs the agent's tools at the waypoint</text>
  <defs>
    <marker id="pg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker>
    <marker id="pr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker>
  </defs>

  <!-- agent -->
  <rect x="20" y="98" width="120" height="54" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="80" y="122" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">agent</text>
  <text x="80" y="139" text-anchor="middle" font-size="9.5" fill="#475569">(kagent)</text>
  <line x1="140" y1="125" x2="206" y2="125" stroke="#334155" stroke-width="1.6" marker-end="url(#pg)"/>

  <!-- waypoint / AccessPolicy -->
  <rect x="206" y="86" width="180" height="78" rx="10" fill="#e0e7ff" stroke="#6366f1" stroke-width="2"/>
  <text x="296" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">agentgateway waypoint</text>
  <text x="296" y="127" text-anchor="middle" font-size="10" fill="#4338ca">kagent AccessPolicy</text>
  <text x="296" y="141" text-anchor="middle" font-size="9.5" fill="#4338ca">action: ALLOW</text>
  <text x="296" y="153" text-anchor="middle" font-size="9.5" fill="#4338ca">tools: [ sum ]</text>

  <!-- tools -->
  <line x1="386" y1="112" x2="500" y2="100" stroke="#16a34a" stroke-width="2" marker-end="url(#pg)"/>
  <rect x="500" y="82" width="200" height="30" rx="7" fill="#dcfce7" stroke="#16a34a"/>
  <text x="600" y="102" text-anchor="middle" font-size="11" fill="#14532d">sum   ✓ allowed</text>
  <line x1="386" y1="140" x2="500" y2="152" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#pr)"/>
  <rect x="500" y="132" width="200" height="42" rx="7" fill="#fee2e2" stroke="#dc2626"/>
  <text x="600" y="150" text-anchor="middle" font-size="10.5" fill="#7f1d1d">echo · printenv · reverse_text</text>
  <text x="600" y="165" text-anchor="middle" font-size="10.5" fill="#7f1d1d">to_uppercase   ✗ denied</text>

  <text x="360" y="212" text-anchor="middle" font-size="11" fill="#64748b">The allowlist names one tool; every other tool on that server is filtered from the agent's list and cannot be called.</text>
</svg></div>

`accesspolicy-on.sh` applies the AccessPolicy, labels the MCP server for the waypoint (the kmcp translator provisions the waypoint Gateway + route), and restarts the agent so it re-lists through the waypoint.

In [ ]:
./scripts/accesspolicy-on.sh

<details>
<summary><strong>What got created (click to expand): one AccessPolicy becomes an agentgateway policy</strong></summary>

`accesspolicy-on.sh` applies one high-level `AccessPolicy` (the platform owner's intent), labels the MCP server for the waypoint, and restarts the agent. You write this:

```yaml
apiVersion: policy.kagent-enterprise.solo.io/v1alpha1
kind: AccessPolicy
metadata: { name: allow-sum-only, namespace: kagent }
spec:
  action: ALLOW
  from:
    subjects: [{ kind: Agent, name: agentdemo, namespace: kagent }]
  targetRef:
    kind: MCPServer
    name: everything-server
    tools: [sum]
```

The kmcp translator turns that into a concrete `EnterpriseAgentgatewayPolicy` on the everything-server waypoint, compiling the subject and tool into a CEL rule over the agent's proven identity:

```yaml
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: accesspolicy-allow-sum-only-waypoint, namespace: kagent }   # translator-provisioned
spec:
  targetRefs: [{ group: agentgateway.dev, kind: AgentgatewayBackend, name: everything-server }]
  backend:
    mcp:
      authorization:
        action: Allow
        policy:
          matchExpressions:
          - (source.identity.namespace == "kagent" && source.identity.serviceAccount == "agentdemo") && (mcp.tool.name == "sum")
```

So "agentdemo may call only `sum`" becomes an enforced CEL check keyed on the agent's SPIFFE identity and the MCP tool name. Denied tools are filtered from the agent's list at the waypoint, so the agent cannot even see them, let alone call them.

See it live:
```
kubectl --context kind-mesh1 -n kagent get accesspolicy allow-sum-only -o yaml
kubectl --context kind-mesh1 -n kagent get enterpriseagentgatewaypolicy accesspolicy-allow-sum-only-waypoint -o yaml
```

</details>

In [ ]:
echo "  == tools now (everything-server is reduced to sum only) =="
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

In [ ]:
echo "  == the policy is a declarative CR =="
kc -n kagent get accesspolicy allow-sum-only -o yaml | sed -n '1,40p'

Revert the governance so the agent sees the full tool list again (leaves the demo re-runnable):

In [ ]:
./scripts/accesspolicy-off.sh

## 7. Turn a REST API into MCP tools (no code)

Not every tool is a purpose-built MCP server. agentgateway reads a plain **OpenAPI** spec and exposes **every operation as an MCP tool**, deriving each tool's input schema from the operation's parameters and body. We point it at the public Swagger Petstore, and its REST operations become callable MCP tools, with the same governance as any other backend.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 240" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="720" height="240" rx="10" fill="#f8fafc"/>
  <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Turn a REST API into governed MCP tools (OpenAPI → MCP)</text>
  <defs><marker id="oa" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>

  <!-- REST API -->
  <rect x="20" y="92" width="120" height="56" rx="9" fill="#e2e8f0" stroke="#64748b"/>
  <text x="80" y="114" text-anchor="middle" font-size="12" font-weight="600" fill="#334155">Petstore</text>
  <text x="80" y="131" text-anchor="middle" font-size="9.5" fill="#475569">REST + OpenAPI</text>
  <text x="180" y="110" text-anchor="middle" font-size="9" fill="#475569">spec</text>
  <line x1="140" y1="120" x2="226" y2="120" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>

  <!-- agentgateway -->
  <rect x="226" y="82" width="200" height="76" rx="10" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/>
  <text x="326" y="106" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">agentgateway</text>
  <text x="326" y="123" text-anchor="middle" font-size="10" fill="#166534">EnterpriseAgentgatewayBackend</text>
  <text x="326" y="137" text-anchor="middle" font-size="9.5" fill="#166534">static.protocol: OpenAPI</text>
  <text x="326" y="150" text-anchor="middle" font-size="9.5" fill="#166534">each REST op → an MCP tool</text>
  <text x="470" y="110" text-anchor="middle" font-size="9" fill="#475569">MCP /mcp</text>
  <line x1="426" y1="120" x2="512" y2="120" stroke="#334155" stroke-width="1.6" marker-end="url(#oa)"/>

  <!-- agent -->
  <rect x="512" y="92" width="188" height="56" rx="9" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="606" y="114" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">agent / MCP client</text>
  <text x="606" y="131" text-anchor="middle" font-size="9.5" fill="#475569">calls getPetById, findByStatus…</text>

  <text x="360" y="202" text-anchor="middle" font-size="11" fill="#64748b">No code: agentgateway reads the OpenAPI spec and exposes each operation as an MCP tool the agent can call, with the same governance.</text>
</svg></div>

In [ ]:
echo "  == 1) load the Petstore's live OpenAPI spec into a ConfigMap =="
curl -s https://petstore3.swagger.io/api/v3/openapi.json \
  | kc -n agentgateway-system create configmap petstore-openapi --from-file=schema=/dev/stdin \
      --dry-run=client -o yaml \
  | kc apply -f -

In [ ]:
echo "  == 2) expose it as MCP with protocol: OpenAPI =="
kc -n agentgateway-system apply -f - <<EOF
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: petstore-api, namespace: agentgateway-system }
spec:
  entMcp:
    sessionRouting: Stateless          # a REST API has no MCP session
    failureMode: FailClosed
    targets:
    - name: petstore
      static:
        host: petstore3.swagger.io      # the public Swagger Petstore API
        port: 443
        protocol: OpenAPI               # parse the schema, one tool per operation
        openAPI:
          schemaRef: { name: petstore-openapi }
        policies:
          tls:
            sni: petstore3.swagger.io   # originate TLS to the public HTTPS API
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: petstore-mcp, namespace: agentgateway-system }
spec:
  parentRefs: [{ name: ar-ingress }]
  hostnames: ["petstore.${LB}.sslip.io"]
  rules:
  - backendRefs: [{ group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: petstore-api }]
EOF
sleep 5

In [ ]:
MCP="http://petstore.${LB}.sslip.io/"
echo "  == 3) the REST operations are now MCP tools =="
for i in $(seq 1 15); do
  OUT=$(curl -s -m5 -X POST "$MCP" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" \
        -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}')
  echo "$OUT" | grep -q '"name"' && break; sleep 3
done
echo "$OUT" | sed 's/^data: //' | grep -oE '"name":"[^"]+"' | sed 's/"name":/  - /;s/"//g'

In [ ]:
MCP="http://petstore.${LB}.sslip.io/"
call() { curl -s -m8 -X POST "$MCP" -H "Content-Type: application/json" -H "Accept: application/json,text/event-stream" -d "$1" | sed 's/^data: //'; }
echo "  == 4) call them, real Petstore data over MCP =="
echo "  -- addPet (creates pet 777) --"
call '{"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":"addPet","arguments":{"body":{"id":777,"name":"Demo Dog","status":"available","photoUrls":["u"]}}}}' \
  | grep -oE '"structuredContent":\{[^}]*\}' | head -1
echo "  -- getPetById 777 (read it back) --"
call '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"getPetById","arguments":{"path":{"petId":777}}}}' \
  | grep -oE '"structuredContent":\{[^}]*\}' | head -1

## 8 · MCP Code Mode: solve the quadratic in one turn, not eleven

**The problem with standard tool calling.** Give the model a calculator MCP server and ask it to solve `x² − 5x + 6 = 0`. In **standard mode** it calls one tool at a time — `mul`, `sub`, `sqrt`, `add`, `div` — **eleven** sequential round-trips, and because every turn re-sends the whole history the token cost climbs each step (≈ 8,400 tokens here).

**Code mode.** Flip **one field** on the MCP backend and agentgateway stops handing the model eleven tools; instead it exposes a single `run_code` tool over the same calculator functions. The model writes a short JavaScript program that runs all eleven operations **locally in one turn** and returns `x = {2,3}` — ≈ 650 tokens, no growing history.

**The field:** `toolMode` on the `EnterpriseAgentgatewayBackend` MCP backend, `Standard` (default) → `Code`. (`Search` exposes a searchable tool index; `CodeSearch` does both; `codeMode` may only be set when `toolMode` is `Code` or `CodeSearch`.) Same calculator server, same governance, one field.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 336" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="760" height="336" rx="12" fill="#0f172a"/><text x="30" y="42" font-size="21" font-weight="700" fill="#e2e8f0" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">Tool Calling: Standard Mode</text><rect x="46" y="78" width="180" height="52" rx="9" fill="#1e293b" stroke="#475569"/><text x="136" y="99" text-anchor="middle" font-size="12" fill="#cbd5e1">Client</text><text x="136" y="116" text-anchor="middle" font-size="9" fill="#94a3b8">(User, Agent, Harness)</text><line x1="136" y1="130" x2="136" y2="158" stroke="#6366f1" stroke-width="1.4"/><circle cx="136" cy="176" r="19" fill="#1e293b" stroke="#f59e0b" stroke-width="2"/><text x="136" y="181" text-anchor="middle" font-size="15" font-weight="700" fill="#f59e0b">11</text><line x1="136" y1="195" x2="136" y2="206" stroke="#6366f1" stroke-width="1.4"/><line x1="86" y1="206" x2="186" y2="206" stroke="#6366f1" stroke-width="1.4"/><line x1="86" y1="206" x2="86" y2="220" stroke="#6366f1" stroke-width="1.4"/><line x1="186" y1="206" x2="186" y2="220" stroke="#6366f1" stroke-width="1.4"/><rect x="40" y="220" width="92" height="86" rx="9" fill="#1e293b" stroke="#475569"/><text x="86" y="238" text-anchor="middle" font-size="10.5" fill="#cbd5e1">Tools</text><text x="57" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">+</text><text x="86" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">-</text><text x="115" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">×</text><text x="57" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">÷</text><text x="86" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">√</text><text x="115" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">π</text><text x="57" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">log</text><text x="86" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">sin</text><text x="115" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">xʸ</text><rect x="140" y="220" width="92" height="86" rx="9" fill="#1e293b" stroke="#475569"/><text x="186" y="240" text-anchor="middle" font-size="10.5" fill="#cbd5e1">LLM</text><text x="186" y="272" text-anchor="middle" font-size="11" font-weight="700" fill="#e2e8f0">✳ Claude</text><text x="404" y="90" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">1</text><text x="416" y="90" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">mul(-5,-5)</text><rect x="558" y="80" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="80" width="62" height="11" rx="3" fill="#c2703f"/><text x="742" y="90" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">450</text><text x="404" y="109" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">2</text><text x="416" y="109" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">mul(4,1)</text><rect x="558" y="99" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="99" width="70" height="11" rx="3" fill="#c2703f"/><text x="742" y="109" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">510</text><text x="404" y="128" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">3</text><text x="416" y="128" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">mul(4,6)</text><rect x="558" y="118" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="118" width="79" height="11" rx="3" fill="#c2703f"/><text x="742" y="128" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">575</text><text x="404" y="147" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">4</text><text x="416" y="147" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">sub(25,24)</text><rect x="558" y="137" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="137" width="88" height="11" rx="3" fill="#c2703f"/><text x="742" y="147" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">640</text><text x="404" y="166" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">5</text><text x="416" y="166" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">sqrt(1)</text><rect x="558" y="156" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="156" width="97" height="11" rx="3" fill="#c2703f"/><text x="742" y="166" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">700</text><text x="404" y="185" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">6</text><text x="416" y="185" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">mul(-1,-5)</text><rect x="558" y="175" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="175" width="106" height="11" rx="3" fill="#c2703f"/><text x="742" y="185" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">765</text><text x="404" y="204" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">7</text><text x="416" y="204" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">add(5,1)</text><rect x="558" y="194" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="194" width="114" height="11" rx="3" fill="#c2703f"/><text x="742" y="204" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">825</text><text x="404" y="223" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">8</text><text x="416" y="223" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">sub(5,1)</text><rect x="558" y="213" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="213" width="123" height="11" rx="3" fill="#c2703f"/><text x="742" y="223" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">890</text><text x="404" y="242" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">9</text><text x="416" y="242" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">mul(2,1)</text><rect x="558" y="232" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="232" width="131" height="11" rx="3" fill="#c2703f"/><text x="742" y="242" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">950</text><text x="404" y="261" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">10</text><text x="416" y="261" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">div(6,2)</text><rect x="558" y="251" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="251" width="140" height="11" rx="3" fill="#c2703f"/><text x="742" y="261" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">1015</text><text x="404" y="280" text-anchor="end" font-size="9.5" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">11</text><text x="416" y="280" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">div(4,2)</text><rect x="558" y="270" width="150" height="11" rx="3" fill="#1e293b" stroke="#334155"/><rect x="558" y="270" width="150" height="11" rx="3" fill="#c2703f"/><text x="742" y="280" text-anchor="end" font-size="9" fill="#94a3b8" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">1080</text><rect x="416" y="300" width="326" height="26" rx="7" fill="#fbcfcf" stroke="#dc2626"/><text x="470" y="317" font-size="12" font-weight="700" fill="#7f1d1d" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">8,400 tokens</text><text x="565" y="317" font-size="9.5" font-style="italic" fill="#7f1d1d">Every turn re-sends the full history</text></svg></div>

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 348" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="760" height="348" rx="12" fill="#0f172a"/><text x="30" y="42" font-size="21" font-weight="700" fill="#e2e8f0" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">Tool Calling: Code Mode</text><rect x="46" y="78" width="180" height="52" rx="9" fill="#1e293b" stroke="#475569"/><text x="136" y="99" text-anchor="middle" font-size="12" fill="#cbd5e1">Client</text><text x="136" y="116" text-anchor="middle" font-size="9" fill="#94a3b8">(User, Agent, Harness)</text><line x1="136" y1="130" x2="136" y2="158" stroke="#6366f1" stroke-width="1.4"/><circle cx="136" cy="176" r="19" fill="#1e293b" stroke="#16a34a" stroke-width="2"/><text x="136" y="181" text-anchor="middle" font-size="15" font-weight="700" fill="#16a34a">1</text><line x1="136" y1="195" x2="136" y2="206" stroke="#6366f1" stroke-width="1.4"/><line x1="86" y1="206" x2="186" y2="206" stroke="#6366f1" stroke-width="1.4"/><line x1="86" y1="206" x2="86" y2="220" stroke="#6366f1" stroke-width="1.4"/><line x1="186" y1="206" x2="186" y2="220" stroke="#6366f1" stroke-width="1.4"/><rect x="40" y="220" width="92" height="86" rx="9" fill="#1e293b" stroke="#475569"/><text x="86" y="238" text-anchor="middle" font-size="10.5" fill="#cbd5e1">Tools</text><text x="57" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">+</text><text x="86" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">-</text><text x="115" y="258" text-anchor="middle" font-size="8.5" fill="#94a3b8">×</text><text x="57" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">÷</text><text x="86" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">√</text><text x="115" y="276" text-anchor="middle" font-size="8.5" fill="#94a3b8">π</text><text x="57" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">log</text><text x="86" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">sin</text><text x="115" y="294" text-anchor="middle" font-size="8.5" fill="#94a3b8">xʸ</text><rect x="140" y="220" width="92" height="86" rx="9" fill="#1e293b" stroke="#475569"/><text x="186" y="240" text-anchor="middle" font-size="10.5" fill="#cbd5e1">LLM</text><text x="186" y="272" text-anchor="middle" font-size="11" font-weight="700" fill="#e2e8f0">✳ Claude</text><rect x="406" y="66" width="330" height="28" rx="7" fill="#1e293b" stroke="#64748b"/><text x="571" y="85" text-anchor="middle" font-size="12" fill="#e2e8f0" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">“Solve x² − 5x + 6 = 0”</text><rect x="406" y="102" width="330" height="196" rx="9" fill="#0f2a1e" stroke="#16a34a" stroke-width="1.6"/><text x="420" y="123" font-size="12.5" font-weight="700" fill="#86efac" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">run_code()</text><text x="726" y="123" text-anchor="end" font-size="9" fill="#4ade80">11 ops run locally · 1 turn</text><rect x="418" y="132" width="306" height="158" rx="6" fill="#0b1f16" stroke="#166534"/><text x="420" y="148" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">const t1 = mul(b, b),</text><text x="716" y="148" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 1</text><text x="452" y="160.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">t2 = mul(4, a),</text><text x="716" y="160.5" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 2</text><text x="452" y="173.0" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">t3 = mul(t2, c),</text><text x="716" y="173.0" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 3</text><text x="452" y="185.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">d  = sub(t1, t3),</text><text x="716" y="185.5" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 4</text><text x="452" y="198.0" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">r  = sqrt(d),</text><text x="716" y="198.0" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 5</text><text x="452" y="210.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">nb = mul(-1, b),</text><text x="716" y="210.5" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 6</text><text x="452" y="223.0" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">n1 = add(nb, r),</text><text x="716" y="223.0" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 7</text><text x="452" y="235.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">n2 = sub(nb, r),</text><text x="716" y="235.5" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 8</text><text x="452" y="248.0" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">dn = mul(2, a),</text><text x="716" y="248.0" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 9</text><text x="452" y="260.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">x1 = div(n1, dn),</text><text x="716" y="260.5" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 10</text><text x="452" y="273.0" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">x2 = div(n2, dn);</text><text x="716" y="273.0" text-anchor="end" font-size="9" fill="#4b5563" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">// 11</text><text x="420" y="285.5" font-size="9.5" fill="#cbd5e1" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">return [x1, x2];</text><rect x="406" y="306" width="330" height="16" rx="5" fill="#1e293b" stroke="#64748b"/><text x="571" y="318" text-anchor="middle" font-size="11" fill="#e2e8f0" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">x = {2,3}</text><rect x="406" y="326" width="330" height="18" rx="5" fill="#bbf7d0" stroke="#16a34a"/><text x="452" y="339" font-size="11" font-weight="700" fill="#14532d" font-family="ui-monospace,SFMono-Regular,Menlo,Consolas,monospace">650 tokens</text><text x="527" y="339" font-size="9" font-style="italic" fill="#14532d">Single-turn with no history</text></svg></div>

### Run it live: a calculator MCP server, Standard vs Code

We front a small **calculator MCP server** (`add / sub / mul / div / sqrt / pow`) with agentgateway and solve `x² − 5x + 6 = 0` two ways. A tiny MCP client (`demo-scripts/mcp-solve.py`) drives it and reports how many **client-visible** tool calls each mode takes.

In [ ]:
: "${CTX:=kind-mesh1}"
LAB="$(git rev-parse --show-toplevel)/istio-ambient-demo-kind"
# build the calculator MCP server (streamable HTTP) and push to the kind registry
docker build -q -t localhost:5001/calc-mcp:latest "$LAB/demo-scripts/calc-mcp" && docker push -q localhost:5001/calc-mcp:latest
kubectl --context $CTX apply -f "$LAB/demo-scripts/yaml-codemode/calc-mcp.yaml"
kubectl --context $CTX -n agentgateway-system rollout status deploy/calc-mcp --timeout=120s

In [ ]:
: "${CTX:=kind-mesh1}"
# front the calculator with an MCP backend (toolMode: Standard) + a route on the AR ingress
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: { name: calc-mcp, namespace: agentgateway-system }
spec:
  entMcp:
    toolMode: Standard          # each calculator tool exposed individually
    sessionRouting: Stateful
    targets:
    - name: calc
      static: { host: calc-mcp.agentgateway-system.svc.cluster.local, port: 3000, protocol: StreamableHTTP, path: /mcp }
EOF
LB=$(kubectl --context $CTX -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
kubectl --context $CTX apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: calc-mcp, namespace: agentgateway-system }
spec:
  parentRefs: [{ name: ar-ingress }]
  hostnames: ["calc.${LB}.sslip.io"]
  rules:
  - backendRefs: [{ group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: calc-mcp }]
EOF
sleep 5
echo "MCP endpoint fronted by agentgateway:  http://calc.${LB}.sslip.io/"

**Standard mode** — the client makes one tool call per operation (eleven round-trips, history re-sent each turn):

In [ ]:
: "${CTX:=kind-mesh1}"
LAB="$(git rev-parse --show-toplevel)/istio-ambient-demo-kind"
LB=$(kubectl --context $CTX -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
pkill -f 'port-forward.*svc/ar-ingress 8099' 2>/dev/null || true
nohup kubectl --context $CTX -n agentgateway-system port-forward svc/ar-ingress 8099:80 >/tmp/ar-ingress-pf.log 2>&1 &
sleep 3
python3 "$LAB/demo-scripts/mcp-solve.py" --url http://localhost:8099/ --host "calc.${LB}.sslip.io"

**Code mode** — flip **one field**, `toolMode: Code`. The gateway now exposes a single `run_code` tool; the client solves the whole quadratic in **one** call:

In [ ]:
: "${CTX:=kind-mesh1}"
LAB="$(git rev-parse --show-toplevel)/istio-ambient-demo-kind"
kubectl --context $CTX -n agentgateway-system patch enterpriseagentgatewaybackend calc-mcp --type=merge -p '{"spec":{"entMcp":{"toolMode":"Code"}}}'
sleep 6
LB=$(kubectl --context $CTX -n agentgateway-system get gateway ar-ingress -o jsonpath='{.status.addresses[0].value}')
pkill -f 'port-forward.*svc/ar-ingress 8099' 2>/dev/null || true
nohup kubectl --context $CTX -n agentgateway-system port-forward svc/ar-ingress 8099:80 >/tmp/ar-ingress-pf.log 2>&1 &
sleep 3
python3 "$LAB/demo-scripts/mcp-solve.py" --url http://localhost:8099/ --host "calc.${LB}.sslip.io"

**Tear down:** `kubectl --context kind-mesh1 -n agentgateway-system delete enterpriseagentgatewaybackend/calc-mcp httproute/calc-mcp deploy/calc-mcp svc/calc-mcp`

## 9. Invoke the same agent on AWS Bedrock AgentCore (runtime #2)

You kicked off the AgentCore deploy back in §3b; while you worked through kagent, governance and the MCP steps, AWS provisioned the *identical* governed agent as a second runtime. `arctl get runtimes` shows `aws-agentcore` alongside `kind-kagent`, and `arctl get agents` shows the `agentdemo-agentcore` record the deploy published (same source, ECR image).

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 230" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"><rect x="0" y="0" width="720" height="230" rx="10" fill="#f8fafc"/><text x="360" y="24" text-anchor="middle" font-size="14.5" font-weight="700" fill="#0f172a">9 · One agent record, two runtimes</text><defs><marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker><marker id="d" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#d97706"/></marker><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#2563eb"/></marker><marker id="n" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs><rect x="250" y="58" width="220" height="60" rx="8" fill="#e0e7ff" stroke="#6366f1" stroke-width="1.6"/><text x="360" y="80" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">agentdemo</text><text x="360" y="98" text-anchor="middle" font-size="8.5" fill="#4338ca">one Agent record in the registry</text><line x1="300" y1="118" x2="180" y2="158" stroke="#16a34a" stroke-width="1.8" marker-end="url(#g)"/><text x="214" y="132" text-anchor="middle" font-size="8" font-weight="700" fill="#166534">deploy</text><line x1="420" y1="118" x2="548" y2="158" stroke="#d97706" stroke-width="1.8" marker-end="url(#d)"/><text x="506" y="132" text-anchor="middle" font-size="8" font-weight="700" fill="#92400e">deploy (no rewrite)</text><rect x="60" y="160" width="240" height="52" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.6"/><text x="180" y="182" text-anchor="middle" font-size="11" font-weight="700" fill="#14532d">kind-kagent</text><text x="180" y="198" text-anchor="middle" font-size="8" fill="#166534">in-cluster runtime (done in §4)</text><rect x="430" y="160" width="240" height="52" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.6"/><text x="550" y="182" text-anchor="middle" font-size="11" font-weight="700" fill="#7c2d12">aws-agentcore</text><text x="550" y="198" text-anchor="middle" font-size="8" fill="#92400e">AWS Bedrock AgentCore · us-east-1</text><text x="360" y="224" text-anchor="middle" font-size="10.5" fill="#64748b">The same agentdemo record deploys to whichever runtime the platform allows: in-cluster kind-kagent, or AWS Bedrock AgentCore, no rewrite.</text></svg></div> Ask it to roll the dice — this runs in **AWS**, not your cluster. `ac-invoke.sh` waits for the runtime to be **READY** first (the background deploy takes a few minutes from §3b), so if it is still finishing it pauses a moment, then prints the answer.

> Still deploying? Watch it: `tail -f /tmp/agentcore-deploy.log`

In [ ]:
arctl get runtimes
echo
./scripts/ac-invoke.sh "Roll a 13-sided die, add 48291 to the result, then tell me if it is prime."

## Reset / teardown

To hand `mesh1` back to the other labs and reclaim about 1.9 GiB, park Part 4's platform with `bash scripts/reset.sh down` from `istio-ambient-demo-kind/demo-scripts/agentregistry`. The Reset cell near the top brings it back up next time. The reset also removes the deployed AgentCore runtime instance in AWS (the `aws-agentcore` platform connection, the cross-account role and the ECR repo stay). To remove the whole AgentRegistry platform (kagent, AgentRegistry, Keycloak) from `mesh1`, tear the demo suite down with `./demo-scripts/setup.sh teardown` from `istio-ambient-demo-kind`.